# Regenerate one quadrant's S2 + SR cache

Standalone notebook for redoing the S2 monthly fetch + OpenSR pass on a **single** quadrant. Use when the main `pipeline.ipynb` driver flags a duplicate / truncated cache (e.g. `rayong_nw` containing NE data, or a smoke-test artefact under `rayong_sw`).

Workflow:
1. Set `QUADRANT` in the config cell below to `NW` / `NE` / `SW` / `SE`.
2. Run top-to-bottom.
3. The S2 monthly cache lands under `data/_cache/s2_monthly/rayong_<q>/` and the SR cache under `data/_cache/s2_sr/rayong_<q>/`.
4. Re-open `pipeline.ipynb` and re-run `§5 driver` -- the new cache is picked up automatically.

**Delete the old cache directory first** if you want a clean rebuild:
```
rm -rf data/_cache/s2_sr/rayong_nw
rm -rf data/_cache/s2_monthly/rayong_nw   # only if you also want fresh openEO
```

In [ ]:
from __future__ import annotations
import sys, warnings
from pathlib import Path
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import rasterio
import rioxarray as rxr
import xarray as xr

warnings.filterwarnings("ignore", category=UserWarning)
print("python:", sys.version.split()[0])

In [ ]:
# >>> SELECT THE QUADRANT TO REBUILD HERE <<<
QUADRANT = "NW"                # NW / NE / SW / SE
TIME_START = "2024-10-01"
TIME_END   = "2024-12-31"
SR_STEPS   = 50                # diffusion sampling steps; 25 = smoke, 50 = full
FORCE_FETCH = False            # set True to re-download S2 even if the cache exists

# Rayong province bbox, split at the centroid -- mirrors lib/rayong.ts and
# pipeline.ipynb's QUADRANT_BBOX dict.
_W, _S, _E, _N = 100.9845, 12.5834, 101.8305, 13.1635
_CLNG, _CLAT   = 101.4291, 12.8539
QUADRANT_BBOX = {
    "FULL": (_W,    _S,    _E,    _N   ),
    "NW":   (_W,    _CLAT, _CLNG, _N   ),
    "NE":   (_CLNG, _CLAT, _E,    _N   ),
    "SW":   (_W,    _S,    _CLNG, _CLAT),
    "SE":   (_CLNG, _S,    _E,    _CLAT),
}
if QUADRANT not in QUADRANT_BBOX:
    raise ValueError(f"QUADRANT must be one of {list(QUADRANT_BBOX)}, got {QUADRANT!r}")

REPO_ROOT  = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_ROOT  = REPO_ROOT / "data"
CACHE_ROOT = DATA_ROOT / "_cache"
AOI_BBOX   = QUADRANT_BBOX[QUADRANT]
AOI_NAME   = f"rayong_{QUADRANT.lower()}"

BANDS_10M  = ("B02", "B03", "B04", "B08")
BANDS_20M  = ("B05", "B06", "B07", "B11", "B12")
SCL_BAND   = "SCL"
SR_SCALE   = 4

S2_DIR = CACHE_ROOT / "s2_monthly" / AOI_NAME
SR_DIR = CACHE_ROOT / "s2_sr"      / AOI_NAME
S2_DIR.mkdir(parents=True, exist_ok=True)
SR_DIR.mkdir(parents=True, exist_ok=True)

print(f"QUADRANT = {QUADRANT}")
print(f"  bbox     {AOI_BBOX}")
print(f"  aoi_name {AOI_NAME}")
print(f"  time     {TIME_START} -> {TIME_END}")
print(f"  s2 cache {S2_DIR}")
print(f"  sr cache {SR_DIR}")

## 1 · S2 monthly · cloud-masked median via CDSE openEO

OIDC login the first time; subsequent runs reuse the token. The fetch buffers the bbox by ~500 m so reprojection rounding does not strip slivers at the edges, and explicitly resamples every contributing MGRS tile to UTM 47N before the temporal reducer runs (without this, openEO has been seen to clip overlap regions to the source CRS of whichever tile it saw first).

In [ ]:
import openeo

S2_TARGET_CRS = "EPSG:32647"
S2_EDGE_BUFFER_DEG = 0.005

cached_tifs = sorted(S2_DIR.glob("*.tif"))
if cached_tifs and not FORCE_FETCH:
    print(f"S2 cache hit · {len(cached_tifs)} files in {S2_DIR}")
else:
    CONN = openeo.connect("openeo.dataspace.copernicus.eu")
    try:
        CONN.authenticate_oidc()
    except Exception as e:
        print(f"openEO auth failed: {e}")
        raise

    w, s, e, n = AOI_BBOX
    b = S2_EDGE_BUFFER_DEG
    bbox = dict(west=w-b, south=s-b, east=e+b, north=n+b, crs="EPSG:4326")
    print(f"openEO request bbox (buffered +{b}°): {bbox}")
    bands = list(BANDS_10M) + list(BANDS_20M) + [SCL_BAND]

    cube = CONN.load_collection(
        "SENTINEL2_L2A",
        spatial_extent=bbox,
        temporal_extent=[TIME_START, TIME_END],
        bands=bands,
        max_cloud_cover=85,
    )
    scl = cube.band(SCL_BAND)
    cube = cube.mask((scl == 3) | (scl == 8) | (scl == 9) | (scl == 10))
    cube = cube.filter_bands(list(BANDS_10M) + list(BANDS_20M))
    cube = cube.resample_spatial(
        resolution=10,
        method="bilinear",
        projection=int(S2_TARGET_CRS.split(":")[1]),
    )
    cube = cube.aggregate_temporal_period(period="month", reducer="median")

    job = cube.execute_batch(title=f"S2 monthly · {AOI_NAME}", out_format="GTiff")
    job.get_results().download_files(str(S2_DIR))
    print(f"S2 monthly tifs downloaded under {S2_DIR}")

# Load the stack so the SR cell can iterate over its time coord.
def _load_s2_stack(s2_dir: Path) -> xr.DataArray:
    tifs = sorted(s2_dir.glob("*.tif"))
    window_start = pd.to_datetime(TIME_START).replace(day=1)
    window_end   = pd.to_datetime(TIME_END)
    arrs, kept, skipped = [], [], []
    for p in tifs:
        month = "".join(c for c in p.stem if c.isdigit())[:6]
        if len(month) < 6:
            continue
        ts = pd.to_datetime(month, format="%Y%m")
        if ts < window_start or ts > window_end:
            skipped.append(str(ts.date())[:7]); continue
        a = rxr.open_rasterio(p, masked=True).astype("float32")
        a = a.expand_dims(time=[ts])
        arrs.append(a); kept.append(str(ts.date())[:7])
    if not arrs:
        raise RuntimeError(f"No cached S2 months inside [{TIME_START}, {TIME_END}] under {s2_dir}")
    stack = xr.concat(arrs, dim="time").rename({"band": "band_idx"})
    print(f"S2 stack {stack.shape} · crs {stack.rio.crs}")
    print(f"  months kept   : {kept}")
    if skipped: print(f"  months skipped: {skipped} (outside window)")
    return stack

S2 = _load_s2_stack(S2_DIR)

## 2 · OpenSR · 4x super-resolution on B02/B03/B04/B08

`opensr_model.SRLatentDiffusion` on the 10 m bands. Tile size 128 LR -> 512 SR. Cached per month under `data/_cache/s2_sr/rayong_<q>/sr_YYYYMM.tif`; existing tifs are skipped so a partial rerun only redoes missing months.

In [ ]:
import torch
import opensr_model
from omegaconf import OmegaConf
from io import StringIO
import requests

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} · device {DEVICE}")

_SR_CFG_URL = "https://raw.githubusercontent.com/ESAOpenSR/opensr-model/refs/heads/main/opensr_model/configs/config_10m.yaml"
_SR_MODEL = None

def get_sr_model():
    global _SR_MODEL
    if _SR_MODEL is None:
        cfg_yaml = requests.get(_SR_CFG_URL, timeout=30).text
        cfg = OmegaConf.load(StringIO(cfg_yaml))
        _SR_MODEL = opensr_model.SRLatentDiffusion(cfg, device=DEVICE)
        _SR_MODEL.load_pretrained(cfg.ckpt_version)
        print(f"opensr-model loaded · ckpt {cfg.ckpt_version}")
    return _SR_MODEL

In [ ]:
@torch.no_grad()
def super_resolve_month(month_arr: xr.DataArray, tile: int = 128, overlap: int = 32, steps: int = SR_STEPS, desc: str = "tiles") -> xr.DataArray:
    """Tile-wise SR. Model expects (B, 4, 128, 128) and returns 4x upsampled."""
    model = get_sr_model()
    rgb_nir = month_arr.isel(band_idx=slice(0, 4)).values.astype("float32") / 10000.0
    C, H, W = rgb_nir.shape
    out = np.zeros((C, H * SR_SCALE, W * SR_SCALE), dtype="float32")
    cnt = np.zeros_like(out[0])

    step = tile - overlap
    coords = [(y, x) for y in range(0, H, step) for x in range(0, W, step)]
    for (y, x) in tqdm(coords, desc=desc, leave=False, mininterval=1.0):
        patch = rgb_nir[:, y:y+tile, x:x+tile]
        ph, pw = patch.shape[1], patch.shape[2]
        if ph < 16 or pw < 16: continue
        if ph < tile or pw < tile:
            pad = np.zeros((C, tile, tile), dtype="float32")
            pad[:, :ph, :pw] = patch
            patch = pad
        t = torch.from_numpy(patch).unsqueeze(0).to(DEVICE)
        sr = model.forward(t, sampling_steps=steps).squeeze(0).cpu().numpy()
        sr = sr[:, : ph * SR_SCALE, : pw * SR_SCALE]
        ys, xs = y * SR_SCALE, x * SR_SCALE
        out[:, ys:ys+sr.shape[1], xs:xs+sr.shape[2]] += sr
        cnt[ys:ys+sr.shape[1], xs:xs+sr.shape[2]] += 1
    out /= np.maximum(cnt[None], 1)

    sr_da = xr.DataArray(out, dims=("band_idx", "y", "x"), coords={"band_idx": list(range(C))})
    sr_da = sr_da.rio.write_crs(month_arr.rio.crs)
    sr_da = sr_da.rio.write_transform(month_arr.rio.transform() * rasterio.Affine.scale(1.0 / SR_SCALE))
    return sr_da


months = list(S2.time.values)
for t in tqdm(months, desc=f"SR · {AOI_NAME}"):
    label   = pd.to_datetime(t).strftime("%Y-%m")
    cache_p = SR_DIR / f"sr_{pd.to_datetime(t).strftime('%Y%m')}.tif"
    if cache_p.exists():
        tqdm.write(f"[{label}] cache hit · {cache_p.name}")
        continue
    tqdm.write(f"[{label}] super-resolving (steps={SR_STEPS}) …")
    sr = super_resolve_month(S2.sel(time=t), desc=f"{label} tiles")
    sr.rio.to_raster(cache_p, compress="DEFLATE", tiled=True)
    tqdm.write(f"[{label}] wrote {cache_p.name}")

print(f"\nSR cache for {AOI_NAME} now under {SR_DIR}:")
for p in sorted(SR_DIR.glob("*.tif")):
    print(f"  {p.name}")

## 3 · Verify the regenerated cache

Reads back one SR tif and prints its shape + bounds + CRS. Compares against the expected quadrant bbox (reprojected to UTM 47N) so you can spot the dup-cache / smoke-test footprints that caused the problem in the first place.

In [ ]:
from pyproj import Transformer

tifs = sorted(SR_DIR.glob("sr_*.tif"))
if not tifs:
    raise RuntimeError(f"no SR tifs under {SR_DIR}")

sr = rxr.open_rasterio(tifs[0], masked=True)
actual_bounds = tuple(round(b, 1) for b in sr.rio.bounds())
print(f"SR cache: {tifs[0].name}")
print(f"  shape  {tuple(sr.shape)}")
print(f"  crs    {sr.rio.crs}")
print(f"  bounds {actual_bounds}")

# Expected UTM 47N bounds for the selected quadrant
w, s, e, n = AOI_BBOX
tf = Transformer.from_crs("EPSG:4326", "EPSG:32647", always_xy=True)
minx, miny = tf.transform(w, s)
maxx, maxy = tf.transform(e, n)
expected = (round(minx, 1), round(miny, 1), round(maxx, 1), round(maxy, 1))
print(f"  expected (UTM 47N) approx {expected}")

def _close(a, b, tol_m: float = 5000.0) -> bool:
    return all(abs(x - y) < tol_m for x, y in zip(a, b))

if _close(actual_bounds, expected):
    print("\n  OK: SR cache bounds match the selected quadrant (within ±5 km tolerance).")
else:
    print("\n  WARNING: SR cache bounds do not match the selected quadrant!")
    print("  Likely cause: stale tif in the directory. Remove the SR_DIR contents and rerun.")